# 04 - Transformations
## Greater London House Price Analysis

Este notebook documenta el proceso de visualización de la capa silver del dataset **UK Price Paid** 
filtrado para Greater London con las transformaciones necesarias para la creación de la capa gold del dataset.

### Objetivo
Visualizar los datos disponibles en la tabla Silver, para realizar transformaciones que aporten valor al dataset, 
guardándose en la última capa (gold).

### Tabla de entrada
`workspace.default.london_silver` — 2,384,979 filas · 21 columnas

### Pipeline
```
Bronze (raw) → Silver (limpia) → Silver enriquecida (nulos tratados) --> Transformaciones (nuevas variables/KPI´s) --> Capa Gold 
```


## 1. Visualización de los datos de la capa Silver (enriquecida)

Visualizamos las primeras cinco líneas de esta capa silver, para confirmar sus dimensiones y variables. Seguidamente realizaremos distintos gráficos sobre las variables disponibles en la capa silver, para facilitar su comprensión y nos permita saber dónde y cómo debemos enriquecer los datos para la capa final gold.

In [0]:
df_silver = spark.table("workspace.default.london_silver")
display(df_silver.limit(5))

#### Gráfico de Ventas totales por año:
Visualizamos graficamente la evolución de ventas totales por en el área de Greater london


In [0]:
from pyspark.sql.functions import col, count

# 1. Agregación directa usando las columnas existentes en Silver
df_counts_by_year = df_silver \
    .filter((col("county") == "GREATER LONDON") & 
            (col("year") >= 2005) & 
            (col("year") <= 2025)) \
    .groupBy("year") \
    .agg(count("*").alias("total_sales")) \
    .orderBy("year")

# 2. Visualización
display(df_counts_by_year)

Databricks visualization. Run in Databricks to view.

#### GRÁFICOS:
- Nº DE VENTAS TOTALES POR RANGO DE PRECIO
- Nº DE VENTAS TOTALES POR TIPO DE INMUEBLE
- Nº DE VENTAS TOTALES POR TIPO DE OBRA



In [0]:
from pyspark.sql.functions import col, year, count, when

# Asumiendo que tu DataFrame de la capa silver se llama 'df_silver'
# y que tiene una columna 'price' numérica.

# 1. Definimos los rangos de precio (price_range) para Londres
# Esto crea la variable categórica necesaria para el gráfico.
df_gold_pre = df_silver \
    .filter((col("county") == "GREATER LONDON") & 
            (year(col("date_of_transfer")) >= 2005) & 
            (year(col("date_of_transfer")) <= 2025)) \
    .withColumn("price_range", 
                when(col("price") < 250000, "< £250k")
                .when((col("price") >= 250000) & (col("price") < 500000), "£250k - £500k")
                .when((col("price") >= 500000) & (col("price") < 750000), "£500k - £750k")
                .when((col("price") >= 750000) & (col("price") < 1000000), "£750k - £1M")
                .otherwise("> £1M"))

# 2. Realizamos UNA única agregación masiva en Spark
# Esto cuenta las ventas para todas las combinaciones únicas de tus 4 variables.
df_combined_categorical_gold = df_gold_pre \
    .groupBy("property_type", "old_new", "property_type_desc", "price_range") \
    .agg(count("*").alias("total_sales"))

# 3. Usamos display() sobre el dataframe combinado.
# Esto genera una sola tabla de salida, pero te permite crear múltiples gráficos en pestañas.
display(df_combined_categorical_gold)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

#### GRÁFICOS: 
- Nº VENTAS TOTALES POR MES
- Nº VENTAS TOTALES POR CUATRIMESTRE

In [0]:
from pyspark.sql.functions import col

# Creamos un solo DataFrame con toda la jerarquía temporal
df_time_gold = df_silver \
    .filter((col("county") == "GREATER LONDON") & (col("year") >= 2005)) \
    .groupBy("year", "quarter", "month") \
    .count() \
    .withColumnRenamed("count", "total_sales") \
    .orderBy("year", "quarter", "month")

# Ahora sí, lanzamos el display
display(df_time_gold)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

#### VENTAS TOTALES POR DISTRITO:

In [0]:
from pyspark.sql.functions import col, desc

# Agrupamos por la variable 'district' que ya tenemos en Silver
df_sales_by_district = df_silver \
    .filter(col("year") >= 2005) \
    .groupBy("district") \
    .count() \
    .withColumnRenamed("count", "total_sales") \
    .orderBy(desc("total_sales"))

# Mostramos el resultado para configurar el gráfico
display(df_sales_by_district)

Databricks visualization. Run in Databricks to view.

## 2. TRANSFORMACIONES PARA ENRIQUECIMIENTO DE LA CAPA GOLD: 
Se realiza una pre-selección de las variables de la capa silver y se decide no utilizar las variables paon, saon, ppd_category y record_status dado que se considera que no aportan valor suficiente al consumidor final. 


In [0]:
from pyspark.sql.functions import col, round, when, avg
from pyspark.sql.window import Window

# 1. Creamos la base con todas las columnas originales que pediste recuperar
df_gold = df_silver.filter((col("county") == "GREATER LONDON") & (col("year") >= 2005) & (col("year") <= 2025)) \
    .select(
        "transaction_id", "price", "date_of_transfer", "postcode", "street", 
        "locality", "town_city", "district", "year", "month", "quarter", 
        "property_type_desc", "price_range", "old_new", "duration", "county"
    )

# Guardado inicial
df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.london_gold")
print("Base Gold creada con todas las columnas originales.")

#### A). TRANSFORMACIÓN: PRECIO MEDIO ANUAL
Se calcula el precio medio anual de las viviendas para cada año del dataset

In [0]:
# Cálculo
win_year = Window.partitionBy("year")
df_gold = df_gold.withColumn("avg_price_annual", round(avg("price").over(win_year), 0))

# Guardado
df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.london_gold")


> #### GRÁFICO EVOLUCIÓN PRECIO MEDIO ANUAL:

In [0]:
# Visualización 1: Evolución General del Precio en Londres
print("Variable 'avg_price_annual' añadida.")
display(df_gold.select("year", "avg_price_annual").distinct().orderBy("year"))

Databricks visualization. Run in Databricks to view.

#### B). TRANSFORMACIÓN: PRECIO MEDIO ANUAL POR TIPO DE INMUEBLE 
Se calucla el precio medio anual en función del tipo de inmueble.

In [0]:
# Cálculo
win_type = Window.partitionBy("year", "property_type_desc")
df_gold = df_gold.withColumn("avg_price_by_type", round(avg("price").over(win_type), 0))

# Guardado
df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.london_gold")


#### GRÁFICO PRECIO MEDIO ANUAL POR TIPO DE INMUEBLE:

In [0]:
# Visualización 2: Evolución por Tipo de Propiedad
print("Variable 'avg_price_by_type' añadida.")
display(df_gold.select("year", "property_type_desc", "avg_price_by_type").distinct().orderBy("year", "property_type_desc"))

Databricks visualization. Run in Databricks to view.

#### C). TRANSFORMACIÓN: PRECIO MEDIO POR DISTRITO y % DE DESVIACIÓN
Se calcula el precio medio en función del distrito donde se encuentre el inmueble

In [0]:
# Cálculo
win_district = Window.partitionBy("year", "district")
df_gold = df_gold.withColumn("avg_price_by_district", round(avg("price").over(win_district), 0)) \
                 .withColumn("price_vs_district_pct", round((col("price") / col("avg_price_by_district") - 1) * 100, 2))

# Guardado
df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.london_gold")


#### GRÁFICO PRECIO MEDIO POR DISTRITO:

In [0]:
# Visualización 3: Ranking de Distritos más caros (Datos de 2024)
print("Variables 'avg_price_by_district' y 'price_vs_district_pct' añadidas.")
display(df_gold.filter(col("year") == 2024).select("district", "avg_price_by_district").distinct().orderBy(col("avg_price_by_district").desc()))

Databricks visualization. Run in Databricks to view.

#### D). TRANSFORMACIÓN: PRECIO MEDIO POR TIPO DE OBRA
Se calcula el precio medio en función si la obra es de nueva construcción o no

In [0]:
# Primero recuperamos/creamos la columna de status que faltaba
df_gold = df_gold.withColumn("property_status", when(df_silver.old_new == "Y", "New Build").otherwise("Established Property"))

# Cálculo
win_status = Window.partitionBy("year", "property_status")
df_gold = df_gold.withColumn("avg_price_by_status", round(avg("price").over(win_status), 0))

# Guardado Final
df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.london_gold")


#### GRÁFICO EVOLUCIÓN PRECIO MEDIO POR TIPO DE OBRA:

In [0]:
# Visualización 4: Comparativa Obra Nueva vs Usada
print("Variable 'avg_price_by_status' añadida. Capa Gold finalizada.")
display(df_gold.select("year", "property_status", "avg_price_by_status").distinct().orderBy("year", "property_status"))

Databricks visualization. Run in Databricks to view.

### 3. GUARDADO FINAL DE LA CAPA GOLD
Comprobamos que las variables creadas y las que se han seleccionado de la capa silver se han guardado correctamente

In [0]:
df_gold = spark.table("workspace.default.london_gold")
display(df_gold.limit(5))
print("Tabla london_gold actualizada")
print(f"   Filas totales:  {df_gold.count():,}")
print(f"   Columnas:       {len(df_gold.columns)}")


### Tabla final
`workspace.default.london_gold` — **2,384,979 filas y 22 columnas** ✅